# オプション課題：ViT が得意な画像を集めて，転移学習で確かめる

**問い**：ViT が CNN（ResNet18）より得意な画像はどんな画像か．仮説を立てて写真を集め，実際に差が出るかを 5 シードで確かめる．

- 手がかり：CNN は小さな窓（3×3 など）で **近くの画素** を見て，層を重ねて少しずつ広く見る．ViT は画像を 16×16 のパッチに切り，自己注意で **最初の層から全部のパッチどうし** の関係を見る
- 第4回の写真（`assets/mydata/`）と新しく集めた写真の両方で，ResNet18 凍結と ViT 凍結を比べる
- 提出は任意．写真を撮る時間が要るので，授業のあとに取り組んでよい

## (0) 番号の設定

In [ ]:
# ===== (0) 番号の設定 =====
MEMBER_NO = 1                 # ← グループ内の自分の番号 (1〜5)．presenter はこの番号の順に演習ごとに交代する

# 授業用フォルダ（AI_TD）のルートを import パスに追加する（ノートブックをどこから開いても動く）
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".dlcourse_root").exists())
sys.path.insert(0, str(ROOT))
print("作業フォルダ:", ROOT)

## (1) 仮説（写真を集める前に書く）

このセルをダブルクリックして，＿＿ に書き込む．

- ViT が得意だと考える画像の特徴：＿＿
- そう考えた理由（CNN と ViT の見方の違いから）：＿＿
- 集める 2 つのクラス（具体的に）：＿＿
- 予想：ViT 凍結 − ResNet18 凍結 の差は，第4回の写真で ＿＿ ポイント，新しい写真で ＿＿ ポイント

## (2) 写真を集める

1. 2 つのクラスを **各 50 枚以上** 撮る（少ないとテストが数枚になり，1 枚で 10 ポイント近く動く）（第4回と同じく，顔・個人情報は写さない．自分で撮ったものだけ）
2. `assets/myphotos_vit/<クラス名>/` に入れる（フォルダ名は半角英数字）
3. ターミナルで `python setup/prepare_mydata.py --vit` を実行する（`assets/mydata_vit/` にそろう）

## (3) 2 つのデータ × 2 つのモデルを 5 回ずつ回す（教員機で約 30 秒．ノート PC では数分）

In [ ]:
import pandas as pd
from common import ASSETS
from common.device import get_device
from common.logger import ResultLogger
from common.mydata import image_dir, diff_ratio, verdict
from common.transfer import run

device = get_device()
DATASETS = {"第4回の写真": image_dir(), "ViT 用に集めた写真": ASSETS / "mydata_vit"}
if not DATASETS["ViT 用に集めた写真"].exists():
    raise RuntimeError("assets/mydata_vit/ が無い．(2) の手順で写真をそろえる")
logger = ResultLogger(None, f"member{MEMBER_NO}", course="c1", day="d5", exercise="opt5", device=device)
rows = []
for data_name, root in DATASETS.items():
    for kind in ["ResNet18", "ViT"]:
        for seed in range(5):
            r = run(kind, "freeze", seed=seed, root=root, device=device)
            logger.log_many({k: r[k] for k in ["test_acc", "val_acc_best", "train_sec"]}, condition=f"{root.name}_{kind}_freeze", seed=seed)
            rows.append({"データ": data_name, "モデル": f"{kind} 凍結", "シード": seed, "テスト正解率": r["test_acc"] * 100})
        print(f"{data_name}・{kind} 凍結 終了（テスト {r['n_test']} 枚．1 枚で {100 / r['n_test']:.1f} ポイント）")
df = pd.DataFrame(rows)

## (4) 表と判定

In [ ]:
print(df.groupby(["データ", "モデル"], sort=False)["テスト正解率"].agg(平均="mean", 標準偏差="std").round(1).to_string())
print()
for data_name in DATASETS:
    t = df[df["データ"] == data_name].pivot(index="シード", columns="モデル", values="テスト正解率")
    d, s, q = diff_ratio(t["ResNet18 凍結"], t["ViT 凍結"])
    print(f"{data_name}：ResNet18 → ViT 差 {d:+.1f} ポイント ÷ 標準偏差 {s:.1f} = {q:.1f} → {verdict(q)}")

## (5) 考察（任意で提出）

1. 仮説は確かめられたか．(4) の差 ÷ 標準偏差 を根拠に書く：＿＿
2. 予想と違った所と，その理由として考えられること：＿＿
3. 仮説を確かめ直すなら，次はどんな写真をどう集めるか：＿＿